In [17]:
from IPython.display import HTML, display

html_content = """
<div style='font-family:Segoe UI;padding:20px;background:#f8f9fa;border-radius:10px;'>

<h2 style='color:#4285F4;'>📚 Gemini-Powered AI Book Assistant</h2>

<p>
This application reads a PDF book, stores its knowledge in a searchable memory,
and uses Google's Gemini model to answer questions, create summaries, generate ideas,
and provide insights based on the book content.
</p>

<h3>📖 What Happens?</h3>

<ol>
<li>Load API keys and connect to Gemini AI.</li>
<li>Read and extract text from the PDF book.</li>
<li>Store the book content in ChromaDB (AI memory).</li>
<li>Create a Retrieval Agent (RAG Agent).</li>
<li>User asks questions about the book.</li>
<li>RAG Agent finds relevant content from the book.</li>
<li>Gemini analyzes the retrieved content.</li>
<li>Gemini generates answers, summaries, ideas, or code.</li>
<li>AgentOps monitors the entire workflow.</li>
</ol>

<h3>🔍 Example Use Cases</h3>

<ul>
<li>Summarize the entire book.</li>
<li>Perform calculations mentioned in the book.</li>
<li>Generate AI & Data Science marketing ideas.</li>
<li>Create brainstorming concepts.</li>
<li>Generate Python code when needed.</li>
</ul>

<h3>⚙️ Components Used</h3>

<table style='border-collapse:collapse;width:100%;background:white;'>
<tr style='background:#e9ecef;'>
<th style='padding:10px;border:1px solid #ccc;'>Component</th>
<th style='padding:10px;border:1px solid #ccc;'>Purpose</th>
</tr>

<tr>
<td style='padding:10px;border:1px solid #ccc;'>PDF Book</td>
<td style='padding:10px;border:1px solid #ccc;'>Source of knowledge</td>
</tr>

<tr>
<td style='padding:10px;border:1px solid #ccc;'>ChromaDB</td>
<td style='padding:10px;border:1px solid #ccc;'>Stores searchable book content</td>
</tr>

<tr>
<td style='padding:10px;border:1px solid #ccc;'>RetrieveUserProxyAgent</td>
<td style='padding:10px;border:1px solid #ccc;'>Searches the book before answering</td>
</tr>

<tr>
<td style='padding:10px;border:1px solid #ccc;'>Gemini Model</td>
<td style='padding:10px;border:1px solid #ccc;'>Understands content and generates answers</td>
</tr>

<tr>
<td style='padding:10px;border:1px solid #ccc;'>AgentOps</td>
<td style='padding:10px;border:1px solid #ccc;'>Tracks and monitors AI activity</td>
</tr>

</table>

<br>

<h3>🔄 Workflow</h3>

<pre style='background:white;padding:15px;border-radius:8px;'>

PDF Book
   │
   ▼
Extract Text
   │
   ▼
Store in ChromaDB
   │
   ▼
RAG Retrieval Agent
   │
   ▼
Find Relevant Content
   │
   ▼
Gemini AI
   │
   ▼
Generate Answer / Summary / Ideas
   │
   ▼
Show Result to User
   │
   ▼
AgentOps Monitoring

</pre>

<div style='background:#d1e7dd;padding:15px;border-radius:8px;'>
✅ Simple Summary:<br><br>
The system reads a PDF book, saves its knowledge in ChromaDB, searches relevant
sections when a question is asked, and uses Google Gemini to generate intelligent
answers, summaries, marketing ideas, brainstorming content, and code based on
the book's content.
</div>

</div>
"""

display(HTML(html_content))

Component,Purpose
PDF Book,Source of knowledge
ChromaDB,Stores searchable book content
RetrieveUserProxyAgent,Searches the book before answering
Gemini Model,Understands content and generates answers
AgentOps,Tracks and monitors AI activity


# Setting base for Retrive Chat

In [2]:
import json
import os

import chromadb

import autogen
from autogen import AssistantAgent
from autogen.agentchat.contrib.retrieve_user_proxy_agent import RetrieveUserProxyAgent

# Accepted file formats for that can be stored in
# a vector database instance
from autogen.retrieve_utils import TEXT_FORMATS

config_list = autogen.config_list_from_json("OAI_CONFIG_LIST.json")

assert len(config_list) > 0
print("models to use: ", [config_list[i]["model"] for i in range(len(config_list))])

Patching name='__init__', member=<function BedrockClient.__init__ at 0x0000024443B85620>, patched=<function function.__call__ at 0x0000024443B85580>
Patching name='_retries', member=5, patched=5
Patching name='cost', member=<function BedrockClient.cost at 0x0000024443B85940>, patched=<function function.__call__ at 0x0000024443B85BC0>
Patching name='create', member=<function BedrockClient.create at 0x0000024443B858A0>, patched=<function function.__call__ at 0x0000024443B85C60>
Patching name='get_usage', member=<function BedrockClient.get_usage at 0x0000024443B859E0>, patched=<function function.__call__ at 0x0000024443B85D00>
Patching name='message_retrieval', member=<function BedrockClient.message_retrieval at 0x0000024443B856C0>, patched=<function function.__call__ at 0x0000024443B85DA0>
Patching name='parse_custom_params', member=<function BedrockClient.parse_custom_params at 0x0000024443B85760>, patched=<function function.__call__ at 0x0000024443B85E40>
Patching name='parse_params', 

In [3]:
print("Accepted file formats for `docs_path`:")
print(TEXT_FORMATS)

Accepted file formats for `docs_path`:
['txt', 'json', 'csv', 'tsv', 'md', 'html', 'htm', 'rtf', 'rst', 'jsonl', 'log', 'xml', 'yaml', 'yml', 'pdf', 'mdx']


In [4]:
keys = {}

with open("secrets.txt", "r") as f:
    for line in f:
        key, value = line.strip().split("=", 1)
        keys[key] = value

agentops_api_key = keys["AGENTOPS_API_KEY"]
openai_api_key = keys["OPENAI_API_KEY"]
gemini_api_key = keys["GEMINI_API_KEY"]

In [5]:
config_list = [
    {
        "model": "models/gemini-flash-lite-latest",
        "api_key": gemini_api_key,
        "api_type": "google"
    }
]

In [6]:
import agentops
agentops.init(agentops_api_key)

🖇 AgentOps: [OPENAI INSTRUMENTOR] Error setting up OpenAI streaming wrappers: No module named 'openai.resources.beta.chat'
🖇 AgentOps: You're on the agentops free plan 🤔
--- Logging error ---
Traceback (most recent call last):
  File "C:\Users\xsanthkum\AppData\Local\anaconda3\Lib\logging\__init__.py", line 1154, in emit
    stream.write(msg + self.terminator)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\xsanthkum\AppData\Local\anaconda3\Lib\encodings\cp1252.py", line 19, in encode
    return codecs.charmap_encode(input,self.errors,encoding_table)[0]
           ~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeEncodeError: 'charmap' codec can't encode character '\U0001f914' in position 66: character maps to <undefined>
Call stack:
  File "C:\Users\xsanthkum\AppData\Local\anaconda3\Lib\runpy.py", line 198, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "C:\Users\xsanthkum\AppData\Local\anaconda3\Lib\runpy.py", line 88, in _run_code


In [7]:
# 1. create an AssistantAgent instance named "assistant"
assistant = AssistantAgent(
    name="assistant",
    system_message="You are a helpful assistant.",
    llm_config={
        "timeout": 600,
        "cache_seed": 42,
        "config_list": config_list,
    },
)


In [8]:
from autogen.agentchat.contrib.retrieve_user_proxy_agent import RetrieveUserProxyAgent

# Extract text from local PDF files
pdf_files = ["book.pdf"]
#docs_content = [extract_text_from_pdf(file) for file in pdf_files]

from autogen.retrieve_utils import TEXT_FORMATS,extract_text_from_pdf
docs_content= extract_text_from_pdf("book.pdf")

In [9]:
# Define the RetrieveUserProxyAgent instance
ragproxyagent = RetrieveUserProxyAgent(
    name="ragproxyagent",
    human_input_mode="NEVER",
    max_consecutive_auto_reply=3,
    retrieve_config={
        "task": "qa",
        "docs_path": ["autogen-docs/book.pdf"],
        "docs_content": docs_content,  # Provide the extracted content directly
        "chunk_token_size": 200,
        "model": config_list[0]["model"],
        "vector_db": "chroma",
        "overwrite": True,
        "get_or_create": True,
    },
    code_execution_config=False,
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

# Process Starts for Scenario 1 : Generate code based off docstrings w/o human feedback

In [10]:
# reset the assistant. Always reset the assistant before starting a new conversation.
assistant.reset()

qa_problem = "summarize the content and use python code to any calculation given in the book"
chat_result = ragproxyagent.initiate_chat(assistant, message=ragproxyagent.message_generator, problem=qa_problem)
#agentops.end_session("Success")

Trying to create collection.


max_tokens is too small to fit a single line of text. Breaking this line:
	The 22 ...
Failed to split docs with must_break_at_empty_line being True, set to False.
2026-09-17 11:51:04,933 - autogen.agentchat.contrib.retrieve_user_proxy_agent - INFO - Found 215 chunks.
2026-09-17 11:51:04,948 - autogen.agentchat.contrib.vectordb.chromadb - INFO - No content embedding is provided. Will use the VectorDB's embedding function to generate the content embedding.
Model models/gemini-flash-lite-latest not found. Using cl100k_base encoding.


VectorDB returns doc_ids:  [['fe1105cf', '8bf7f8ee', 'f0804281', '1d0894aa', '839a2e04', 'd11d708f', '150c596a', '59e93e46', '1613971f', '7091167f', '43a28a91', '16a1e2eb', 'e7d73411', '02a6f2ee', '451a2a58', 'a5e9b671', '7acf2ce9', 'a1061522', '05dac356', '466a2471']]
Adding content of doc fe1105cf to context.


Model models/gemini-flash-lite-latest not found. Using cl100k_base encoding.


Adding content of doc 8bf7f8ee to context.


...

Adding content of doc 54fb1eca to context.


Model models/gemini-flash-lite-latest not found. Using cl100k_base encoding.


Adding content of doc 8dad8ef3 to context.


Model models/gemini-flash-lite-latest not found. Using cl100k_base encoding.


ragproxyagent (to assistant):

You're a retrieve augmented chatbot. You answer user's questions based on your own knowledge and the
context provided by the user.
If you can't answer the question with or without the current context, you should reply exactly `UPDATE CONTEXT`.
You must give as short an answer as possible.

User's question is: summarize the content and use python code to any calculation given in the book

Context is: product line and focused on overnight package deliv-
ery only.
In a way, the law of leadership—it’ s better to be
first than to be better—enables the first brand or
company to own a word in the mind of the prospect.
But the word the leader owns is so simple that it’ s
invisible.
The leader owns the word that stands for the cat-
egory. For example, IBM owns computer . This is
another way of saying that the brand becomes a
generic name for the category. “We need an IBM
machine.” Is there any doubt that a computer is
being requested?
Y ou can also test the validi

In [11]:
#agentops.end_session("Success")

# Process Starts for Scenario 2 : Answer a question based off docstrings w/o human feedback

In [12]:
# reset the assistant. Always reset the assistant before starting a new conversation.
assistant.reset()

qa_problem = "You are a creative head . do many brainstorms"
chat_result = ragproxyagent.initiate_chat(assistant, message=ragproxyagent.message_generator, problem=qa_problem)

Model models/gemini-flash-lite-latest not found. Using cl100k_base encoding.


VectorDB returns doc_ids:  [['f0804281', '7e1ac0cd', '0e08a976', '81d8712b', 'be843ac3', 'a5e9b671', 'b2f1dbea', 'ba9a27e8', '21ec236c', '51ba5a36', '8126c05a', 'cf32340a', '7e3514a1', '89fd1dac', '5b19ade7', '10e789b5', '451a2a58', 'a1f246ff', '4cf44936', 'e47994d3']]
Adding content of doc f0804281 to context.


Model models/gemini-flash-lite-latest not found. Using cl100k_base encoding.


Adding content of doc 7e1ac0cd to context.


Model models/gemini-flash-lite-latest not found. Using cl100k_base encoding.


...

Adding content of doc 217a91d7 to context.


Model models/gemini-flash-lite-latest not found. Using cl100k_base encoding.


Adding content of doc d0e85c7b to context.


Model models/gemini-flash-lite-latest not found. Using cl100k_base encoding.


ragproxyagent (to assistant):

You're a retrieve augmented chatbot. You answer user's questions based on your own knowledge and the
context provided by the user.
If you can't answer the question with or without the current context, you should reply exactly `UPDATE CONTEXT`.
You must give as short an answer as possible.

User's question is: You are a creative head . do many brainstorms

Context is: you might possibly want to get into.
IBM has erected an enormous computer tent.
Nothing in the computer field, today or in the future,
will fall outside the IBM tent. This is a recipe for
disaster. As new companies, new products, new
ideas invade the computer arena, IBM is going to get
blown away. Y ou can’t defend a rapidly growing mar-
ket like computers even if you are a financial power-
house like IBM. From a strategic point of view, you
74
THE 22 IMMUTABLE LA WS OF MARKETING
22 Laws of Marketing  10/31/02  12:23 PM  Page 74have to be much more selective, picking and choos-
ing the area i

# Process Starts for Scenario 3 : Generate code based off docstrings w/ human feedback

In [15]:
# reset the assistant. Always reset the assistant before starting a new conversation.
assistant.reset()

# set `human_input_mode` to be `ALWAYS`, so the agent will ask for human input at every step.
ragproxyagent.human_input_mode = "ALWAYS"
code_problem = "Based on this book give Marketing idea for AI and Datascience trainig institute. Give 5 ideas. if any coding requires use python and save it as .py"
chat_result = ragproxyagent.initiate_chat(assistant, message=ragproxyagent.message_generator, problem=code_problem)

Model models/gemini-flash-lite-latest not found. Using cl100k_base encoding.


VectorDB returns doc_ids:  [['f0804281', '81d8712b', '89fd1dac', '0e08a976', '150c596a', 'be843ac3', 'ba9a27e8', '8bf7f8ee', '8a2a101c', '039e8163', '451a2a58', 'd0e14ddf', 'b210f301', 'c1b21496', 'd5a791b1', 'a5e9b671', 'fc6e8864', '08e00728', 'ff30ed4d', '788184ee']]
Adding content of doc f0804281 to context.


Model models/gemini-flash-lite-latest not found. Using cl100k_base encoding.


Adding content of doc 81d8712b to context.


Model models/gemini-flash-lite-latest not found. Using cl100k_base encoding.


...

Model models/gemini-flash-lite-latest not found. Using cl100k_base encoding.


ragproxyagent (to assistant):

You're a retrieve augmented chatbot. You answer user's questions based on your own knowledge and the
context provided by the user.
If you can't answer the question with or without the current context, you should reply exactly `UPDATE CONTEXT`.
You must give as short an answer as possible.

User's question is: Based on this book give Marketing idea for AI and Datascience trainig institute. Give 5 ideas. if any coding requires use python and save it as .py

Context is: 123
THE LA W OF ACCELERATION
22 Laws of Marketing  10/31/02  12:23 PM  Page 12322
The Law of 
Resources
Without adequate funding
an idea won’t get off the ground.
124
22 Laws of Marketing  10/31/02  12:23 PM  Page 124If you have a good idea and you’ve picked up this
book with the thought in mind that all you need is a lit-
tle marketing help, this chapter will throw cold water
on that thought.
Even the best idea in the world won’t go very far
without the money to get it off the ground. Invent

Replying as ragproxyagent. Provide feedback to assistant. Press enter to skip and use auto-reply, or type 'exit' to end the conversation:  What can i learn from this book ? Summarize the content


ragproxyagent (to assistant):

What can i learn from this book ? Summarize the content

--------------------------------------------------------------------------------
assistant (to ragproxyagent):

UPDATE CONTEXT

--------------------------------------------------------------------------------


Replying as ragproxyagent. Provide feedback to assistant. Press enter to skip and use auto-reply, or type 'exit' to end the conversation:  exit


# Process Starts for Scenario 4: Answer a question based off docstrings w/ human feedback

In [16]:
# reset the assistant. Always reset the assistant before starting a new conversation.
assistant.reset()

# set `human_input_mode` to be `ALWAYS`, so the agent will ask for human input at every step.
ragproxyagent.human_input_mode = "ALWAYS"
qa_problem = "Based on this book give Marketing idea for AI and Datascience trainig institute. Give 5 ideas. "
chat_result = ragproxyagent.initiate_chat(
    assistant, message=ragproxyagent.message_generator, problem=qa_problem
)  # type "exit" to exit the conversation

Model models/gemini-flash-lite-latest not found. Using cl100k_base encoding.


VectorDB returns doc_ids:  [['f0804281', '81d8712b', '89fd1dac', 'be843ac3', '150c596a', 'b210f301', '8a2a101c', 'ba9a27e8', '0e08a976', '788184ee', '039e8163', '451a2a58', '4e3d050e', 'a1f246ff', 'd5a791b1', '08e00728', '083871f5', 'b99eaa4e', 'c1b21496', 'ff30ed4d']]
Adding content of doc f0804281 to context.


Model models/gemini-flash-lite-latest not found. Using cl100k_base encoding.


Adding content of doc 81d8712b to context.


Model models/gemini-flash-lite-latest not found. Using cl100k_base encoding.


...

ragproxyagent (to assistant):

Summarize the book

--------------------------------------------------------------------------------
assistant (to ragproxyagent):

UPDATE CONTEXT

--------------------------------------------------------------------------------


Replying as ragproxyagent. Provide feedback to assistant. Press enter to skip and use auto-reply, or type 'exit' to end the conversation:  Show important pointers from this book


ragproxyagent (to assistant):

Show important pointers from this book

--------------------------------------------------------------------------------
assistant (to ragproxyagent):

1. Law of Leadership: It’s better to be first than to be better.
2. Law of Perception: Marketing is a battle of perceptions, not products.
3. Law of the Mind: It’s better to get into the mind first than into the marketplace.
4. Law of the Ladder: The strategy to use depends on which rung you occupy on the ladder.
5. Law of Resources: Without adequate funding, an idea won’t get off the ground.

--------------------------------------------------------------------------------


Replying as ragproxyagent. Provide feedback to assistant. Press enter to skip and use auto-reply, or type 'exit' to end the conversation:  



>>>>>>>> NO HUMAN INPUT RECEIVED.
